# Day 11 - Customer Churn Prediction

Predicting customer churn on an imbalanced dataset (22% churn rate). Focus is on properly handling imbalance instead of just reporting accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/churn.csv')
print('Shape:', df.shape)
print('Churn rate:', f"{df['churn'].mean()*100:.1f}%")
df.head()

## 2. Why Accuracy Alone Is Misleading

With 22% churn, predicting "stayed" for everyone gets 78% accuracy while catching zero churners. Need precision/recall per class.

In [ ]:
feature_cols = ['tenure_months', 'monthly_charges', 'total_charges', 'contract_type',
                'support_calls', 'satisfaction_score', 'has_addon_services']

X = df[feature_cols]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Plain vs Balanced Random Forest

In [ ]:
model_plain = RandomForestClassifier(n_estimators=100, random_state=42)
model_plain.fit(X_train_scaled, y_train)
preds_plain = model_plain.predict(X_test_scaled)
print('Plain Random Forest:')
print(classification_report(y_test, preds_plain, target_names=['Stayed', 'Churned']))

In [ ]:
model_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model_balanced.fit(X_train_scaled, y_train)
preds_balanced = model_balanced.predict(X_test_scaled)
print('Balanced Random Forest:')
print(classification_report(y_test, preds_balanced, target_names=['Stayed', 'Churned']))

In [ ]:
auc_plain = roc_auc_score(y_test, model_plain.predict_proba(X_test_scaled)[:, 1])
auc_balanced = roc_auc_score(y_test, model_balanced.predict_proba(X_test_scaled)[:, 1])
print(f'ROC AUC plain:    {auc_plain:.4f}')
print(f'ROC AUC balanced: {auc_balanced:.4f}')

## 4. Feature Importance

In [ ]:
importances = model_balanced.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(9, 5))
plt.bar([feature_cols[i] for i in sorted_idx], importances[sorted_idx], color='#FF5722')
plt.title('Feature Importance')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Wrap-up

Both models scored similarly on ROC AUC (~0.94). Balancing did not meaningfully improve churn recall in this case, which goes against the common assumption that class_weight always helps with imbalance.

This is a useful lesson: always measure the actual effect of a technique rather than applying it automatically. Satisfaction score and contract type were the strongest churn predictors.